# Weights

Said loosely, a **weight** accounts for how much a row in dataset "counts for". There are subtleties, which affect model fitting, statistical inference, bootstrap sampling, and cross validation.

In this notebook, we explore the two types of sample weights that are supported in `py-glm`: *frequency* weights, and *precision* weights. To use weighting correctly, it's important to know what kind of weighting is appropriate for your data and problem.

In [1]:
import numpy as np
import pandas as pd

from glm.datasets import make_frame
from glm.families import Bernoulli, Gaussian
from glm.glm import GLM
from glm.simulation import Simulation, coefficient_table

## Two kinds of weights

`GLM.fit` supports two kinds of weights:

#### Frequency weights 
A row with **frequency weight** `f` stands for `f` *identical observations*. The data table has been compressed, by storing the information in repeated rows one time, with a metadata count. This is the same idea as [run length encoding](https://en.wikipedia.org/wiki/Run-length_encoding) (pedantry: without the sense of ordering implicit in run length encoding). Fitting with frequency weights is *exactly equivalent* to fitting to the equivalent expanded data, with every row repeated `f` times.

#### Precision weights 
A row with **precision weight** `w` is *one observation* of an average. The response is more precise than a single observation's, as an average is more precise than the single observations composing it: its variance is smaller by a factor of `w`:

```
Var[y | X] = φ V(μ) / w
```

For example, a row in our data may represent a single school composed of multiple students. We may not have access to each individual students test scores, but instead have  the school's average test score and the count of students composing that average. The count `w` of students, is an example of a *precision weight*. Another common case is data containing proportions of successes in Bernoulli trials, the number of trials is a precision weight. R's `glm` calls these *prior weights*.

This leads to the following heuristic:

> To tell which you have, ask: **is each row's response a single value, repeated `f` times, or an average over `w` units?**

Both kinds of weights multiply each row's contribution to the deviance, so the same numbers, read as either kind, give the same coefficient estimates. The difference arises in statistical inference, the two kinds of weights imply different semantics about the variance of a single row `var[y | X]`, and so everything downstream of this is changed: confidence intervals and bootstrap estimates.


This table summarizes how each type of weight enters into the various calculations:

| | frequency weights | precision weights |
|---|---|---|
| weight on each row's deviance | `f` | `w` |
| observation count, `n` | `Σ f` | count of rows |
| estimated dispersion `φ` | weighted deviance / `(Σ f − p)` | weighted deviance / `(rows − p)` |
| standardizing columns for ridge | weighted by `f` | not weighted |

Here `p` is the number of estimated coefficients, (`model.p`). 

The two kinds of weights can be used together, appropriate when rows are compressed averages.  In this case each row's weight in the deviance is their product.

## Frequency Weights

Let's imagine we're studying customer's renewing a subscription. We'll sample 2000 customers, each of whom either renewed a subscription, `y = 1`, or did not, `y = 0`, and let probability of renewing depend on two categorical predictors, the customer's `region` (five levels) and their `plan` (two levels):

In [2]:
sim = make_frame(
    Bernoulli(),
    n_samples=2000,
    n_uncorrelated=0,
    categorical={
        "region": ["Plains", "Mountain", "Swamp", "Island", "Forest"], 
        "plan": ["Beginner", "Master"]
    },
    coef_range=(-0.8, 0.8),
    rng=np.random.default_rng(154),
)
customers = sim.data
customers.head()

,region,plan,y
0,Swamp,Beginner,1
1,Island,Master,1
2,Mountain,Master,1
3,Plains,Beginner,0
4,Forest,Beginner,0


This data has a simple structure, we have three columns, all categorical. This makes only:

5 regions × 2 plans × 2 ys = 20 possibilities

possible distinct rows. The order of the rows does not have any meaning here, so the same data can be recorded much more compactly by counting how many times each of the 20 possibilities occurs. We'll fit the same model to the data recorded three ways.

In [3]:
customers_compressed = customers.groupby(["region", "plan", "y"], as_index=False).size()
customers_compressed.head()

,region,plan,y,size
0,Plains,Beginner,0,74
1,Plains,Beginner,1,103
2,Plains,Master,0,70
3,Plains,Master,1,127
4,Mountain,Beginner,0,89


The *size* column here is exactly what we described as a *frequency weight*, and we can fit the same model two ways:

**One row per customer**, unweighted:

In [4]:
formula = "y ~ C(region) + C(plan)"
individual = GLM(Bernoulli()).fit(customers, formula=formula)

**One row per distinct record**: Each row in the compressed data stands for `size` identical customers, so the counts are frequency weights:

In [5]:
frequency = GLM(Bernoulli()).fit(
    customers_compressed, 
    formula=formula, 
    frequency_weights=customers_compressed["size"].to_numpy(float)
)

The two fits agree exactly, in both estimates and standard errors:

In [6]:
fits = {"individual": individual, "frequency": frequency}
pd.concat(
    {
        "estimate": pd.DataFrame({k: m.coef_ for k, m in fits.items()}),
        "std error": pd.DataFrame({k: m.coef_standard_error_ for k, m in fits.items()}),
    },
    axis=1,
).set_axis(individual.X_names).round(4)

estimate            std error          
                      individual frequency individual frequency
Intercept                 0.3370    0.3370     0.1169    0.1169
C(region)[T.Mountain]    -0.0182   -0.0182     0.1475    0.1475
C(region)[T.Swamp]        0.3709    0.3709     0.1543    0.1543
C(region)[T.Island]      -0.5304   -0.5304     0.1427    0.1427
C(region)[T.Forest]       0.5283    0.5283     0.1557    0.1557
C(plan)[T.Master]         0.2526    0.2526     0.0941    0.0941

## Precision Weights

Let's imagine we're studying the effectiveness of some teaching methodology, comparing method `A` to method `B`. Let's sample 2000 students, each of whom took an exam, and achieved a score `y`.  We assume that the teaching method has some effect on the student's expected score, with a nuisance effect of the students `school` (one of five). The scores vary normally around the conditional expectation:

In [7]:
sim = make_frame(
    Gaussian(),
    n_samples=2000,
    n_uncorrelated=0,
    categorical={
        "school": 5, 
        "method": ["A", "B"]
    },
    dispersion=10.0,
    coef_range=(0.0, 10.0),
    rng=np.random.default_rng(154),
)
students = sim.data
students.head()

,school,method,y
0,c,A,15.212349
1,d,B,18.566462
2,b,B,19.259843
3,a,A,10.431198
4,e,A,15.558823


It's common in such studies that data is reported not so granularly, it's quite revealing about the individual student's which is arguably somewhat unethical. Instead, the researchers will receive summarized data, containing the average test score within each level of covariates, and annotated with the size (number of students) within each level:

In [8]:
students_grouped = students.groupby(["school", "method"], as_index=False).agg(
    count=("y", "size"), y=("y", "mean")
)
students_grouped

,school,method,count,y
0,a,A,177,8.330377
1,a,B,197,13.479391
2,b,A,197,12.561613
3,b,B,208,17.457769
4,c,A,198,14.298248
5,c,B,183,20.168828
6,d,A,240,8.769708
7,d,B,207,14.246423
8,e,A,204,14.928286
9,e,B,189,20.386862


Just as before, we can fit the model in two equivalent ways.

**One row per student**, unweighted:

In [9]:
formula = "y ~ C(school) + C(method)"
individual_model = GLM(Gaussian()).fit(students, formula=formula)

**One row per covariate group**, weighted:

In [10]:
precision_model = GLM(Gaussian()).fit(
    students_grouped,
    formula=formula,
    precision_weights=students_grouped["count"].to_numpy(float),
)

Since the response variable here is a *mean*, we use precision weights.

For the purpose of comparison, let's also fit with the count as a frequency weighting:

In [11]:
frequency_model = GLM(Gaussian()).fit(
    students_grouped,
    formula=formula,
    frequency_weights=students_grouped["count"].to_numpy(float),
)

The three fits agree exactly in their estimates, but not in their standard errors:

In [12]:
fits = {"individual": individual_model, "frequency": frequency_model, "precision": precision_model}
pd.concat(
    {
        "estimate": pd.DataFrame({k: m.coef_ for k, m in fits.items()}),
        "std error": pd.DataFrame({k: m.coef_standard_error_ for k, m in fits.items()}),
    },
    axis=1,
).set_axis(individual_model.X_names).round(4)

estimate                      std error                    
               individual frequency precision individual frequency precision
Intercept          8.2144    8.2144    8.2144     0.1769    0.0093    0.2071
C(school)[T.b]     4.1043    4.1043    4.1043     0.2233    0.0117    0.2614
C(school)[T.c]     6.3247    6.3247    6.3247     0.2267    0.0119    0.2654
C(school)[T.d]     0.6051    0.6051    0.6051     0.2183    0.0114    0.2556
C(school)[T.e]     6.7569    6.7569    6.7569     0.2250    0.0118    0.2634
C(method)[T.B]     5.3692    5.3692    5.3692     0.1394    0.0073    0.1632

#### Individual model vs. Precision Weighted

The precision-weighted model's standard errors are all larger than the individual model's, and by the same factor:

In [13]:
(precision_model.coef_standard_error_ / individual_model.coef_standard_error_).round(3)

array([1.171, 1.171, 1.171, 1.171, 1.171, 1.171])

That's a clue. 

To get to the bottom of it we have to inspect the individual ingredients in the standard error computation: the information matrix, and the estimated dispersion.

The two models have the same *information matrix*: 

In [14]:
np.allclose(individual_model.information_matrix_, precision_model.information_matrix_)

True

In the individual model, this matrix is `X'X`, where `X` is the students' design matrix, with 2000 rows. In the precision-weighted model, it's `X_g' D X_g`, where `X_g` is the grouped design matrix, with 10 rows, and `D` is a diagonal matrix of the group counts. These agree because each group's row `x_g` appears `m_g` times among the students' rows:

```
X'X = Σ m_g x_g x_g' = X_g' D X_g
```

So their standard errors can only differ through the other ingredient, the estimated dispersion:

In [15]:
dispersions = pd.Series(
    {"individual": individual_model.dispersion_, "precision": precision_model.dispersion_}
)
print(dispersions.round(2))

individual     9.69
precision     13.28
dtype: float64


These differ, and the square root of their ratio is exactly the ratio of the coefficient standard errors:

In [16]:
print(np.sqrt(dispersions["precision"] / dispersions["individual"]).round(3))

1.171


Both dispersions estimate the same `σ²`, which was supplied as a parameter of our simulation: `dispersion=10.0`, so `σ² = 10`.

The estimate is the model deviance divided by the residual degrees of freedom. But they see different parts of the data. Splitting the sum of each student's residual at their group's average score `ȳ` splits the individual model's deviance into two terms:

```
Σ (y − μ)²   =   Σ (y − ȳ)²   +   Σ m (ȳ − μ)²
 individual      within-group      precision-weighted
  deviance        variation             deviance
```

Expanding the square also gives a cross term:

```
2 Σ (y − ȳ)(ȳ − μ)
```

This vanishes because we can split the sum into those terms coming from a single group, within each of these the `(ȳ − μ)` factor is constant, and the remaining sum is zero as `Σy = mȳ`. This is a standard argument from analysis of variance.

We can confirm all this:

In [17]:
group_average = students.groupby(["school", "method"])["y"].transform("mean")
within_groups = ((students["y"] - group_average) ** 2).sum()

# Equal up to floating point inaccuracies
(individual_model.deviance_ - (within_groups + precision_model.deviance_)).round(2)

np.float64(0.0)

That's how the model deviances shake out, what about the residual degrees of freedom? These split up in the same way: each of the 10 group averages uses up one of the 2000 observations, and the 6 coefficients use up 6 of the 10 averages:

In [18]:
n_groups, p = len(students_grouped), individual_model.p
pd.DataFrame(
    {
        "deviance": [within_groups, precision_model.deviance_, individual_model.deviance_],
        "degrees of freedom": [len(students) - n_groups, n_groups - p, len(students) - p],
    },
    index=["within groups", "between groups (precision model)", "individual model"],
).round(2)

,deviance,degrees of freedom
within groups,19271.86,1990
between groups (precision model),53.14,4
individual model,19325.00,1994


Averaging discards the within-group variation, and with it 1990 of the 1994 degrees of freedom the individual model has for estimating `σ²`. The precision-weighted model estimates `σ²` from the 4 that remain: how the 10 group averages scatter around their fit. So its estimate is much noisier: across repeated samples, it varies around the same `σ²` we provided when simulating the data.

The cost of averaging that *is* reliable shows up in the intervals. When estimating a dispersion, intervals use quantiles of Student's `t` distribution with the residual degrees of freedom, this accounts for the uncertainty in the dispersion. With 4 degrees of freedom, its 97.5th percentile is 2.78, against 1.96 for the individual model, so the precision-weighted intervals are wider, beyond any difference in the standard errors:

In [19]:
widths = pd.DataFrame(
    {
        name: model.summary_frame()["upper 95%"] - model.summary_frame()["lower 95%"]
        for name, model in {"individual": individual_model, "precision": precision_model}.items()
    }
)
widths.round(3)

,individual,precision
Intercept,0.694,1.150
C(school)[T.b],0.876,1.451
C(school)[T.c],0.889,1.474
C(school)[T.d],0.856,1.419
C(school)[T.e],0.882,1.463
C(method)[T.B],0.547,0.906


So averaging loses nothing about the coefficients: the group averages are *sufficient* for them, giving the same estimates and the same information matrix. But it loses most of the information about `σ²`, and the price is wider intervals.

#### Individual Model vs. Frequency Weighted
The frequency weighted standard errors are much smaller than the individual, and this is because they are just incorrect. Using frequency weights in this situation amounts to modeling the case where each student in a school and method group receives the *exact same* test score, with no possibility of between-student variation. It's no surprise we could estimate the effects much more precisely if that were the case.

More precisely, the frequency-weighted model has exactly the same deviance as the precision-weighted one, 53.14: it's the deviance of `m` copies of a student who scored their group's average. The two differ only in the degrees of freedom they divide it by: 2000 − 6 for frequency weights, against 10 − 6 for precision. Frequency weights count the 1990 within-group degrees of freedom as if they carried no variation at all, which is why their dispersion, and standard errors, are far too small.

## Bootstraps with weights

The inference notebook estimated standard errors a second way, with the bootstrap: build many replicate datasets like the original, refit the model to each, and measure directly how the estimates vary. Weights change what a replicate "like the original" is. Getting it wrong distorts the standard errors, just as using the wrong kind of weights (frequency vs. precision) does.

Both bootstraps need the weights (and offsets) the model was fit with. `GLM.fit` records which of them it was given, and the bootstraps raise an error if they're left out, rather than silently bootstrapping a different model.

### The non-parametric bootstrap: resample observations

The non-parametric bootstrap resamples the data. With frequency weights, the object to resample is the *observation*, which is not the row: a row with frequency weight `f` stands for `f` identical customers. To sample correctly, we want to treat each of the `f` customers, though identical, as separate objects capable of being sampled, exactly as if we bootstrapped the expanded data. `py-glm` does this without literally expanding the data. Instead, we draw how many times each row's customers appear in a replicate, from a multinomial distribution, and fit the replicate with these counts as its frequency weights.

So a single resample of the frequency weights looks like this:

In [20]:
np.random.seed(6)
sizes = customers_compressed["size"]
pd.DataFrame(
    {"original": sizes, "resampled": np.random.multinomial(sizes.sum(), sizes / sizes.sum())}
).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
original,74,103,70,127,89,108,69,139,64,134,52,131,127,113,105,102,60,144,47,142
resampled,54,114,65,138,89,109,79,125,55,144,51,132,122,121,104,101,66,140,47,144


This is all encapsulated, you just need to pass along the original weights:

In [21]:
np.random.seed(7)
customer_weights = customers_compressed["size"].to_numpy(float)
replicates = Simulation(frequency).non_parametric_bootstrap(
    customers_compressed, n_sim=500, frequency_weights=customer_weights
)
bootstrap_errors = pd.DataFrame(
    {
        "model": frequency.coef_standard_error_,
        "bootstrap": coefficient_table(replicates).std().to_numpy(),
    },
    index=frequency.X_names,
)
bootstrap_errors.round(3)

,model,bootstrap
Intercept,0.117,0.117
C(region)[T.Mountain],0.147,0.146
C(region)[T.Swamp],0.154,0.157
C(region)[T.Island],0.143,0.141
C(region)[T.Forest],0.156,0.151
C(plan)[T.Master],0.094,0.095


The bootstrap agrees with the model's standard errors.

It would be incorrect to resample the *rows*, each keeping its original weight. Then all of a row's customers are drawn together, or not at all, as if they were one unit. Here, that means bootstrapping 20 records rather than 2000 customers:

In [39]:
rng = np.random.default_rng(8)
whole_row_coefficients, failures = [], 0
for _ in range(500):
    rows = rng.integers(0, len(customers_compressed), size=len(customers_compressed))
    try:
        # Some replicates separate completely, overflowing on the way to
        # failing: silence numpy's warnings about it.
        with np.errstate(all="ignore"):
            model = GLM(Bernoulli()).fit(
                customers_compressed.iloc[rows],
                formula="y ~ C(region) + C(plan)",
                frequency_weights=customer_weights[rows],
            )
        whole_row_coefficients.append(model.coef_)
    except ValueError:
        failures += 1
bootstrap_errors["whole rows"] = np.std(whole_row_coefficients, axis=0, ddof=1)
print(f"{failures} of 500 replicates couldn't be fit")
bootstrap_errors.round(3)

34 of 500 replicates couldn't be fit


,model,bootstrap,whole rows
Intercept,0.117,0.117,7.711
C(region)[T.Mountain],0.147,0.146,9.534
C(region)[T.Swamp],0.154,0.157,9.862
C(region)[T.Island],0.143,0.141,10.231
C(region)[T.Forest],0.156,0.151,9.874
C(plan)[T.Master],0.094,0.095,5.914


Resampling whole rows greatly overstates the uncertainty. Drawing a record's hundred or so customers all at once or not at all makes the replicates far more variable than drawing them individually and independently. Here, with only 20 records, replicates often miss every row of renewals for some region, and then that region's coefficient runs off towards infinity, or can't be estimated at all.

Precision-weighted rows are different: each is *one* observation, an average, so the non-parametric bootstrap resamples each whole, keeping its weight. (Our 10 student groups are too few for that to work here: most replicates would miss some school entirely.)

### The parametric bootstrap: simulate averages

The parametric bootstrap simulates new responses from the conditional distribution `y | X` using the fit model. 

With precision weights, each simulated response must be an *average* of `w` draws, with variance `φ V(μ) / w`, just like the data. To accomplish this, each `ExponentialFamily` implements a sampler from the *sampling distribution of the mean*. For example, the `Gaussian` family samples from a normal distribution with mean `μ` and standard deviation `σ/√w`, which is the sampling distribution of the mean of `w` independent normal random variables. For discrete families, the weights must be whole numbers: the number of draws to average, or for proportions, the number of trials.

For the student groups:

In [23]:
np.random.seed(9)
group_weights = students_grouped["count"].to_numpy(float)
replicates = Simulation(precision_model).parametric_bootstrap(
    students_grouped, n_sim=500, precision_weights=group_weights
)
pd.DataFrame(
    {
        "model": precision_model.coef_standard_error_,
        "parametric bootstrap": coefficient_table(replicates).std().to_numpy(),
    },
    index=precision_model.X_names,
).round(3)

,model,parametric bootstrap
Intercept,0.207,0.213
C(school)[T.b],0.261,0.256
C(school)[T.c],0.265,0.274
C(school)[T.d],0.256,0.250
C(school)[T.e],0.263,0.268
C(method)[T.B],0.163,0.169


The bootstrap agrees with the precision-weighted model, which is all a parametric bootstrap can do: it simulates from the model, with the model's own estimate of the dispersion, so it inherits the noisy estimate from the precision weighted data.

Leaving out the weights from the bootstrap would simulate single students' scores. Since each group has about 200 students, single student scores should have about 200 times the variance of a group average, but the bootstrapped models would fit these unweighted.  For this reason, forgetting to pass weights raises an error instead of misestimating the variance:

In [24]:
try:
    Simulation(precision_model).parametric_bootstrap(students_grouped, n_sim=500)
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: The inputs must match those the model was fit with: the model was fit with precision_weights, so pass them here too.


Frequency weights aren't supported by the parametric bootstrap at all. To simulate the data properly, we would simulate every one of the `f` observations a row stands in for, which needs the expanded data. Simulating just their average would get the coefficients' variation right, but each replicate's dispersion wrong, the same mistake as using frequency weights for averages. There's no cheap way to do this properly, so for now we raise an error. With whole-number frequency weights, simulating every observation is possible, and we may implement it in the future:

In [25]:
try:
    Simulation(frequency).parametric_bootstrap(customers_compressed, n_sim=500)
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: The parametric bootstrap doesn't support models fit with frequency weights. Use the non-parametric bootstrap, which resamples the observations they stand for.


## Summary

To choose weights, ask what each row of your data *is*:

- **A single value, repeated `f` times?** Use `frequency_weights=f`. Fitting is exactly equivalent to fitting the expanded data, with each row repeated `f` times.
- **An average, or a proportion, over `w` units?** Use `precision_weights=w`. Each row is one observation, with variance `φ V(μ) / w`.
- **Both, when rows are compressed averages?** Pass both: each row's weight in the deviance is their product.

The two kinds give the same coefficients, so a mistake doesn't show in the parameter estimates. It shows in the uncertainty: frequency weights count `Σ f` observations, precision weights count rows, and the dispersion, standard errors, and intervals all follow from that count. Using frequency weights for averages makes the standard errors far too small.

Averaging loses nothing about the coefficients, since the group averages are sufficient for their estimation. It does lose the variation within groups, and with it most of the information about the dispersion, which shows up as wider intervals.

When bootstrapping, pass the same weights (and offset) the model was fit with:

- the **non-parametric** bootstrap resamples *observations*: the customers a frequency weight stands for, each separately, and precision-weighted rows whole;
- the **parametric** bootstrap simulates *averages* for precision-weighted rows, and doesn't support frequency weights.

Weights come up again in the regularization notebook, where cross-validation has to decide how to split weighted rows between folds.